Local per-card SAM detection (src/detect_cards.py) measured ~130-230s/image
on CPU at points_per_side=32 (see
tables_figures/eda/sam_card_detection_test/notes.md) -- ~6.5 days serial for
the full 3500-image dataset. This notebook runs the same detection logic on
GPU to measure real per-image runtime, starting with a small LIMIT before
committing to a full run.

**Setup:** upload this notebook directly to Kaggle (kaggle.com/code -> New
Notebook -> File -> Upload Notebook, or via the Kaggle API), attach the
museumscat-specimen-collection-annotation-task competition as Input (Add
Input -> Competition -- no need to upload our own dataset copy), and under
Settings enable Accelerator = GPU T4 x2 (or P100) and Internet = On (needed
to pip install segment-anything and download the SAM checkpoint).

**Output:** for each processed image, one full-resolution crop file per
detected card plus a manifest.csv row (same schema as
src/extract_card_crops.py: crop_filename, source_image_file, card_index, x1,
y1, x2, y2), zipped at the end for download.

## Cell 1: install segment-anything and download the SAM ViT-B checkpoint

In [ ]:
!pip install -q segment-anything
import os
os.makedirs("/kaggle/working/sam_checkpoint", exist_ok=True)
!curl -s -o /kaggle/working/sam_checkpoint/sam_vit_b_01ec64.pth "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
assert os.path.getsize("/kaggle/working/sam_checkpoint/sam_vit_b_01ec64.pth") > 300_000_000

## Cell 2: locate the competition's images (unzip if Kaggle mounted a raw images.zip)

In [ ]:
import os
import zipfile
INPUT_DIR = "/kaggle/input/competitions/museumscat-specimen-collection-annotation-task"
IMAGES_DIR = os.path.join(INPUT_DIR, "images")
if not os.path.isdir(IMAGES_DIR):
    IMAGES_DIR = "/kaggle/working/images"
    os.makedirs(IMAGES_DIR, exist_ok=True)
    with zipfile.ZipFile(os.path.join(INPUT_DIR, "images.zip")) as zf:
        zf.extractall(IMAGES_DIR)
TRAIN_CSV = os.path.join(INPUT_DIR, "train.csv")
TEST_CSV = os.path.join(INPUT_DIR, "test.csv")
assert os.path.isdir(IMAGES_DIR) and os.path.isfile(TRAIN_CSV) and os.path.isfile(TEST_CSV)

## Cell 3: write the per-GPU worker script

In [ ]:
%%writefile sam_worker.py
"""Standalone per-GPU worker for parallel SAM card detection. Kaggle's
GPU T4 x2 setting exposes 2 physical GPUs to the notebook, but a single
Python process only ever uses one of them (SAM/PyTorch don't split a single
inference call across devices on their own). Running this script twice as
two separate OS processes -- each restricted to one physical GPU via
CUDA_VISIBLE_DEVICES, set before torch is even imported -- gets real
simultaneous use of both.

Usage (invoked by the driver cell below, not run directly):
    python sam_worker.py --gpu 0 --image-list image_list_gpu0.txt \
        --images-dir /path/to/images --out-dir /kaggle/working/card_crops
"""
import argparse
import csv
import os

parser = argparse.ArgumentParser()
parser.add_argument("--gpu", type=int, required=True)
parser.add_argument("--image-list", required=True)
parser.add_argument("--images-dir", required=True)
parser.add_argument("--out-dir", required=True)
args = parser.parse_args()

os.environ["CUDA_VISIBLE_DEVICES"] = str(args.gpu)  # must be set before torch is imported

import time
from datetime import timedelta

import cv2
import numpy as np
import torch
from segment_anything import SamAutomaticMaskGenerator, sam_model_registry

CHECKPOINT = "/kaggle/working/sam_checkpoint/sam_vit_b_01ec64.pth"
TARGET_WIDTH = 1600
POINTS_PER_SIDE = 32
MAX_AREA_FRAC = 0.25
MIN_AREA_FRAC = 0.0015
MAX_ASPECT = 4.0
MIN_SOLIDITY = 0.7
CONTAINMENT_THRESH = 0.7

# --- vendored from src/detect_labels.py ---
DASSCO_HSV_LOW = (95, 95, 40)
DASSCO_HSV_HIGH = (130, 255, 255)
DASSCO_OPEN_KERNEL = 15
FURNITURE_X_RATIO = 1.21
BRIGHTNESS_THRESHOLD = 140
OPEN_KERNEL = 21
PADDING = -50
MIN_ROTATION_DEG = 0.05


def _dassco_mask(img):
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, DASSCO_HSV_LOW, DASSCO_HSV_HIGH)
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((DASSCO_OPEN_KERNEL, DASSCO_OPEN_KERNEL), np.uint8))


def _dassco_bbox(img):
    mask = _dassco_mask(img)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    x, y, w, h, _ = stats[largest]
    return (x, y, w, h)


def _dassco_rotation_angle(img):
    mask = _dassco_mask(img)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return 0.0
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    ys, xs = np.where(labels == largest)
    pts = np.column_stack([xs, ys]).astype(np.float32)
    _, _, angle = cv2.minAreaRect(pts)
    while angle <= -45:
        angle += 90
    while angle > 45:
        angle -= 90
    return angle


def _deskew(img):
    angle = _dassco_rotation_angle(img)
    if abs(angle) < MIN_ROTATION_DEG:
        return img, 0.0
    h, w = img.shape[:2]
    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    rotated = cv2.warpAffine(img, matrix, (w, h), flags=cv2.INTER_LINEAR, borderValue=(255, 255, 255))
    return rotated, angle


def find_search_roi(image_path):
    img = cv2.imread(image_path)
    img, angle = _deskew(img)
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    dassco = _dassco_bbox(img)
    safe_cut = int(dassco[0] + FURNITURE_X_RATIO * dassco[2]) if dassco else w // 3
    report_x1 = (dassco[0] + dassco[2]) if dassco else safe_cut

    _, mask = cv2.threshold(gray, BRIGHTNESS_THRESHOLD, 255, cv2.THRESH_BINARY)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((OPEN_KERNEL, OPEN_KERNEL), np.uint8))

    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return angle, (report_x1, 0, w, h)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mount = labels == largest
    mount[:, :safe_cut] = False

    ys, xs = np.where(mount)
    if len(xs) == 0:
        return angle, (report_x1, 0, w, h)

    x1 = max(0, report_x1 - PADDING)
    y1 = max(0, int(ys.min()) - PADDING)
    x2 = min(w, int(xs.max()) + PADDING)
    y2 = min(h, int(ys.max()) + PADDING)
    return angle, (x1, y1, x2, y2)


def get_search_crop(image_path):
    img = cv2.imread(image_path)
    img, _ = _deskew(img)
    _, (x1, y1, x2, y2) = find_search_roi(image_path)
    return img[y1:y2, x1:x2]


# --- vendored from src/detect_cards.py -- device is always "cuda:0" from
# this process's own point of view, since CUDA_VISIBLE_DEVICES above already
# restricted it to a single physical GPU ---

def load_generator():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[gpu {args.gpu}] loading SAM on {device}")
    sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT)
    sam.to(device)
    return SamAutomaticMaskGenerator(sam, points_per_side=POINTS_PER_SIDE)


def downsample(crop_bgr):
    h, w = crop_bgr.shape[:2]
    scale = TARGET_WIDTH / w
    small = cv2.resize(crop_bgr, (TARGET_WIDTH, int(h * scale)))
    return small, scale


def _bbox(mask_entry):
    ys, xs = np.where(mask_entry["segmentation"])
    return int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())


def _overlap_frac(box_a, box_b, area_a):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    if ix2 <= ix1 or iy2 <= iy1:
        return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    return inter / area_a


def select_card_masks(masks, image_shape):
    roi_area = image_shape[0] * image_shape[1]
    candidates = []
    for m in masks:
        area_frac = m["area"] / roi_area
        if area_frac > MAX_AREA_FRAC or area_frac < MIN_AREA_FRAC:
            continue
        x1, y1, x2, y2 = _bbox(m)
        w, h = x2 - x1 + 1, y2 - y1 + 1
        aspect = max(w, h) / max(1, min(w, h))
        if aspect > MAX_ASPECT:
            continue
        bbox_area = w * h
        solidity = m["area"] / bbox_area
        if solidity < MIN_SOLIDITY:
            continue
        candidates.append({"bbox": (x1, y1, x2, y2), "bbox_area": bbox_area})

    candidates.sort(key=lambda c: -c["bbox_area"])
    kept = []
    for c in candidates:
        if any(_overlap_frac(c["bbox"], k["bbox"], c["bbox_area"]) > CONTAINMENT_THRESH for k in kept):
            continue
        kept.append(c)
    return [c["bbox"] for c in kept]


def detect_cards(crop_bgr, generator):
    small, scale = downsample(crop_bgr)
    small_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
    masks = generator.generate(small_rgb)
    small_boxes = select_card_masks(masks, small.shape[:2])

    h, w = crop_bgr.shape[:2]
    full_boxes = []
    for (x1, y1, x2, y2) in small_boxes:
        fx1, fy1 = int(x1 / scale), int(y1 / scale)
        fx2, fy2 = int(x2 / scale), int(y2 / scale)
        full_boxes.append((max(0, fx1), max(0, fy1), min(w, fx2), min(h, fy2)))
    return full_boxes


def main():
    with open(args.image_list, encoding="utf-8") as f:
        image_files = [line.strip() for line in f if line.strip()]

    crops_dir = os.path.join(args.out_dir, "images")
    os.makedirs(crops_dir, exist_ok=True)
    manifest_path = os.path.join(args.out_dir, f"manifest_gpu{args.gpu}.csv")
    processed_log_path = os.path.join(args.out_dir, f"processed_gpu{args.gpu}.log")

    done = set()
    if os.path.exists(processed_log_path):
        with open(processed_log_path, encoding="utf-8") as f:
            done = {line.strip() for line in f if line.strip()}

    todo = [f for f in image_files if f not in done]
    print(f"[gpu{args.gpu}] {len(done)} already done, {len(todo)} to process this run", flush=True)
    if not todo:
        return

    generator = load_generator()
    n_total = len(todo)
    t_start = time.time()

    write_header = not os.path.exists(manifest_path)
    with open(manifest_path, "a", newline="", encoding="utf-8") as mf, \
         open(processed_log_path, "a", encoding="utf-8") as lf:
        writer = csv.writer(mf)
        if write_header:
            writer.writerow(["crop_filename", "source_image_file", "card_index", "x1", "y1", "x2", "y2"])

        for idx, image_file in enumerate(todo, 1):
            crop = get_search_crop(os.path.join(args.images_dir, image_file))
            boxes = detect_cards(crop, generator)

            stem = os.path.splitext(image_file)[0]
            for i, (x1, y1, x2, y2) in enumerate(boxes):
                card_crop = crop[y1:y2, x1:x2]
                crop_filename = f"{stem}_card{i:02d}.jpeg"
                cv2.imwrite(os.path.join(crops_dir, crop_filename), card_crop)
                writer.writerow([crop_filename, image_file, i, x1, y1, x2, y2])
            mf.flush()
            lf.write(image_file + "\n")
            lf.flush()

            elapsed = time.time() - t_start
            avg = elapsed / idx
            elapsed_str = str(timedelta(seconds=int(elapsed)))
            eta = str(timedelta(seconds=int(avg * (n_total - idx))))
            print(f"[gpu{args.gpu}] {idx}/{n_total} images -- {elapsed_str} elapsed, {avg:.2f}s/image, ETA {eta}", flush=True)


if __name__ == "__main__":
    main()


## Cell 4: split the image list in half and run one worker process per GPU

Each `sam_worker.py` invocation is a separate OS process restricted to one physical GPU via `CUDA_VISIBLE_DEVICES` (set inside the script before `torch` is imported), so both run truly in parallel -- confirm with `!nvidia-smi` in another cell while this is running, both GPUs should show utilization.

In [ ]:
import os
import subprocess
import time

import pandas as pd

LIMIT = "train"  # "train" = only train.csv images, "test" = only test.csv images,
                  # None = everything (train then test), or an int = first N images (train first, then test)

OUT_DIR = "/kaggle/working/card_crops"
os.makedirs(OUT_DIR, exist_ok=True)

train_files = pd.read_csv(TRAIN_CSV)["image_file"].tolist()
test_files = pd.read_csv(TEST_CSV)["image_file"].tolist()
on_disk = set(os.listdir(IMAGES_DIR))

if LIMIT == "train":
    todo = [f for f in train_files if f in on_disk]
elif LIMIT == "test":
    todo = [f for f in test_files if f in on_disk]
else:
    ordered = [f for f in train_files + test_files if f in on_disk]
    todo = ordered[:LIMIT] if LIMIT else ordered

half = len(todo) // 2
chunks = [todo[:half], todo[half:]]
for gpu_id, chunk in enumerate(chunks):
    with open(f"image_list_gpu{gpu_id}.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(chunk))
print(f"split {len(todo)} images: gpu0={len(chunks[0])}, gpu1={len(chunks[1])}")

t0 = time.time()
procs = []
logs = []
for gpu_id in range(2):
    log = open(f"gpu{gpu_id}.log", "w")
    logs.append(log)
    p = subprocess.Popen(
        ["python", "sam_worker.py",
         "--gpu", str(gpu_id),
         "--image-list", f"image_list_gpu{gpu_id}.txt",
         "--images-dir", IMAGES_DIR,
         "--out-dir", OUT_DIR],
        stdout=log, stderr=subprocess.STDOUT,
    )
    procs.append(p)

positions = [0, 0]
while any(p.poll() is None for p in procs):
    time.sleep(5)
    for gpu_id in range(2):
        with open(f"gpu{gpu_id}.log") as f:
            f.seek(positions[gpu_id])
            new_output = f.read()
            positions[gpu_id] = f.tell()
        if new_output:
            print(new_output, end="")

for p in procs:
    p.wait()
for gpu_id in range(2):
    with open(f"gpu{gpu_id}.log") as f:
        f.seek(positions[gpu_id])
        new_output = f.read()
    if new_output:
        print(new_output, end="")
for log in logs:
    log.close()
elapsed = time.time() - t0

print(f"both GPU workers finished in {elapsed:.1f}s for {len(todo)} images total")
print(f"-> {elapsed / len(todo):.2f}s/image effective, extrapolated {elapsed / len(todo) * 3500 / 3600:.1f}h for all 3500")
print("--- gpu0.log tail ---")
print("".join(open("gpu0.log").readlines()[-15:]))
print("--- gpu1.log tail ---")
print("".join(open("gpu1.log").readlines()[-15:]))


## Cell 5: merge the two workers' manifests into one

In [ ]:
import csv
import os

manifest_path = os.path.join(OUT_DIR, "manifest.csv")
rows = []
header = None
for gpu_id in range(2):
    part_path = os.path.join(OUT_DIR, f"manifest_gpu{gpu_id}.csv")
    with open(part_path, encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows.extend(reader)

with open(manifest_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(rows)

print(f"merged manifest: {len(rows)} card crops -> {manifest_path}")


## Cell 6: zip the output for download

In [ ]:
import shutil

zip_path = shutil.make_archive("/kaggle/working/card_crops", "zip", "/kaggle/working/card_crops")
print(f"wrote {zip_path}")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("no Colab download API available here -- download the zip manually from the Output pane")
